# Ejercicios de fuerza bruta

**Fuerza bruta** es la estrategia algorítmica más directa: cuando no se conoce (o no se explota) ninguna propiedad especial del problema, se prueban **todas** las posibilidades candidatas y se verifica cada una contra la condición pedida. Es casi siempre la primera solución correcta a la que se llega, y sirve como punto de referencia antes de buscar algo más eficiente.

**Restricción de este notebook:** todas las soluciones son **iterativas** (con ciclos `for`/`while`), sin usar funciones recursivas propias. Cuando se necesita generar combinaciones o permutaciones, se usa el módulo estándar `itertools`, que genera esas secuencias internamente sin que nuestro código recurra.

> **Cómo usar este material:** cada ejercicio trae el enunciado, el **modelo de pensamiento** (el razonamiento que lleva del enunciado al algoritmo), el **análisis de complejidad** (tiempo y espacio, con su justificación), la **solución en Python** con type hints, y **casos de prueba** que validan la solución con `assert`.

Los seis ejercicios están ordenados por dificultad creciente, y también muestran cómo distintos "espacios de búsqueda" llevan a distintas clases de complejidad: desde pares de elementos (n²) hasta permutaciones completas (n!).

## Ejercicio 1: Par que suma un objetivo

### Enunciado

Dada una lista de enteros `nums` y un entero `objetivo`, determina si existen dos índices **distintos** `i` y `j` tales que `nums[i] + nums[j] == objetivo`. Devuelve la primera pareja de índices `(i, j)` que cumpla la condición (con `i < j`), o `None` si no existe ninguna.

### Modelo de pensamiento

1. La fuerza bruta consiste en no asumir ningún atajo: si no se conoce ninguna propiedad especial del problema, la opción más directa es probar **todas** las combinaciones posibles y verificar cada una contra la condición pedida.
2. Aquí, "todas las combinaciones posibles" son todos los pares de índices `(i, j)` con `i < j` — así no comparas un elemento consigo mismo, ni repites el mismo par en el orden contrario.
3. Un ciclo anidado genera esos pares de forma sistemática: el ciclo externo fija `i`, y el ciclo interno recorre `j` desde `i + 1` hasta el final — eso garantiza `i < j` automáticamente, sin necesitar una comprobación adicional.
4. En cuanto encuentras un par que cumple la condición, puedes retornar de inmediato: no hace falta seguir buscando más parejas.

### Complejidad

- **Tiempo:** O(n²) en el peor caso (no existe pareja, o la única pareja válida está al final): el ciclo externo da *n* vueltas y, para cada una, el ciclo interno da hasta *n* vueltas más.
- **Espacio:** O(1) adicional: no se usa ninguna estructura auxiliar que crezca con el tamaño de la entrada (solo variables escalares para los índices).

In [ ]:
def encontrar_par_suma(nums: list[int], objetivo: int) -> tuple[int, int] | None:
    ...

### Casos de prueba

In [ ]:
assert encontrar_par_suma([2, 7, 11, 15], 9) == (0, 1)
assert encontrar_par_suma([3, 2, 4], 6) == (1, 2)
assert encontrar_par_suma([1, 2, 3], 100) is None
assert encontrar_par_suma([5, 5], 10) == (0, 1)
print("Todos los casos de prueba pasaron.")


## Ejercicio 2: Búsqueda de un patrón en un texto

### Enunciado

Dado un texto y un patrón (ambos `str`), encuentra **todas** las posiciones (índices) donde el patrón aparece dentro del texto, probando el patrón contra cada posición de inicio posible, carácter por carácter (sin usar métodos como `str.find`).

### Modelo de pensamiento

1. La fuerza bruta para búsqueda de patrones prueba el patrón en **cada** posición posible de inicio dentro del texto, sin ningún atajo (los atajos son algoritmos más avanzados, como KMP, que se ven en otra estrategia).
2. ¿Cuáles son las posiciones de inicio válidas? Debe haber espacio suficiente para que el patrón completo quepa desde ahí hasta el final del texto: la última posición válida es `len(texto) - len(patron)`.
3. Para cada posición de inicio, compara carácter por carácter el patrón contra el fragmento correspondiente del texto — en cuanto aparece una diferencia, abandona esa posición (con `break`) y pasa a la siguiente; no tiene sentido seguir comparando.
4. Si llegaste a comparar **todos** los caracteres del patrón sin encontrar diferencias, esa posición es una coincidencia real — regístrala.

### Complejidad

- **Tiempo:** O(n·m), siendo *n* el largo del texto y *m* el largo del patrón: hay `n - m + 1` posiciones de inicio, y cada una puede requerir comparar los *m* caracteres del patrón en el peor caso.
- **Espacio:** O(1) adicional para el algoritmo en sí (aparte de la lista de resultados, cuyo tamaño depende de cuántas coincidencias existan y es parte de la salida, no una estructura auxiliar del algoritmo).

In [ ]:
def buscar_patron(texto: str, patron: str) -> list[int]:
    ...

### Casos de prueba

In [ ]:
assert buscar_patron("abababab", "aba") == [0, 2, 4]
assert buscar_patron("aaaa", "aa") == [0, 1, 2]
assert buscar_patron("hola mundo", "xyz") == []
assert buscar_patron("abc", "abcd") == []  # patrón más largo que el texto
print("Todos los casos de prueba pasaron.")


## Ejercicio 3: Máxima suma de un subarreglo contiguo

### Enunciado

Dada una lista de enteros (puede tener negativos, no vacía), encuentra la máxima suma posible de un subarreglo **contiguo** (no vacío). Resuélvelo probando todos los subarreglos posibles, pero evitando recomputar cada suma desde cero.

### Modelo de pensamiento

1. La fuerza bruta "ingenua" sería: para cada posible inicio `i` y cada posible fin `j` (con `i <= j`), sumar los elementos entre `i` y `j` desde cero, y comparar contra el máximo encontrado hasta ahora — eso son tres ciclos anidados (i, j, y uno más para sumar), es decir O(n³).
2. Antes de programar, nota un desperdicio: la suma del subarreglo `[i, j]` y la del subarreglo `[i, j+1]` comparten casi todos los mismos términos — solo cambia un elemento nuevo al final. Recalcular la suma completa cada vez es trabajo repetido innecesario.
3. La mejora (que **sigue siendo fuerza bruta**, porque se siguen probando todos los pares `(i, j)` — solo se evita el ciclo interno de suma): para cada inicio `i`, recorre `j` desde `i` hasta el final, manteniendo una suma acumulada que solo agrega el nuevo elemento `nums[j]` en cada paso.
4. Esto reduce el trabajo de O(n³) a O(n²), sin cambiar la esencia del algoritmo: se siguen revisando *todos* los subarreglos posibles, solo que de forma más eficiente.

### Complejidad

- **Tiempo:** O(n²): dos ciclos anidados (inicio y fin del subarreglo), y cada combinación hace trabajo constante gracias a la suma acumulada (en vez de O(n³) con recomputación completa).
- **Espacio:** O(1) adicional: solo variables escalares (la suma acumulada y el mejor resultado encontrado hasta ahora).

In [ ]:
def maxima_suma_subarreglo(nums: list[int]) -> int:
    ...


### Casos de prueba

In [ ]:
assert maxima_suma_subarreglo([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6  # [4, -1, 2, 1]
assert maxima_suma_subarreglo([1, 2, 3, 4]) == 10  # el arreglo completo
assert maxima_suma_subarreglo([-5, -1, -3]) == -1  # el "menos malo"
assert maxima_suma_subarreglo([5]) == 5
print("Todos los casos de prueba pasaron.")


## Ejercicio 4: Descifrar un código de k dígitos

### Enunciado

Un candado de `k` dígitos (cada uno entre 0 y 9) tiene una combinación secreta. Se te da una función `validar_codigo(codigo: str) -> bool` que indica si un código dado es el correcto. Escribe una función que, por fuerza bruta, encuentre la combinación correcta probando **todos** los códigos posibles de `k` dígitos, en orden, desde `"00...0"` hasta `"99...9"`.

### Modelo de pensamiento

1. Este problema es distinto a los anteriores: la fuerza bruta ya no recorre los datos de entrada (no hay una lista para iterar), sino que recorre el **espacio de soluciones posibles** — todos los códigos que se podrían formar con `k` dígitos.
2. ¿Cuántos códigos posibles hay? Cada dígito tiene 10 valores posibles (0-9), y hay `k` dígitos independientes entre sí → `10^k` combinaciones totales. Ese número va a ser la base del ciclo.
3. Se pueden generar sistemáticamente todos los códigos de `k` dígitos pensando en cada código como un número de 0 a `10^k - 1`, convertido a texto con ceros a la izquierda — así, un solo `for numero in range(10 ** k)` recorre todas las combinaciones, sin necesitar `k` ciclos anidados (uno por dígito).
4. Para cada candidato, formatea el número como texto de `k` dígitos (rellenando con ceros a la izquierda) y pruébalo con `validar_codigo`. En cuanto encuentres el correcto, retorna de inmediato.
5. Esta es una complejidad **exponencial** en `k` — muy distinta a los O(n²) anteriores, que eran polinomiales sobre el tamaño de la entrada. Aquí `k` (la cantidad de dígitos) aparece en el **exponente**, lo que hace que el trabajo crezca muchísimo más rápido incluso para valores pequeños de `k`.

### Complejidad

- **Tiempo:** O(k · 10^k): se prueban hasta 10^k códigos en el peor caso (no existe, o es el último), y formatear/comparar cada código candidato de `k` dígitos cuesta O(k).
- **Espacio:** O(k) adicional: el texto del código candidato tiene `k` caracteres en cada iteración (no se acumula nada más).

In [ ]:
from typing import Callable


def descifrar_codigo(k: int, validar_codigo: Callable[[str], bool]) -> str | None:
    ...


### Casos de prueba

In [ ]:
from typing import Callable


def hacer_validador(secreto: str) -> Callable[[str], bool]:
    def validar(codigo: str) -> bool:
        return codigo == secreto
    return validar

assert descifrar_codigo(3, hacer_validador("042")) == "042"
assert descifrar_codigo(2, hacer_validador("99")) == "99"
assert descifrar_codigo(4, hacer_validador("0007")) == "0007"
assert descifrar_codigo(2, hacer_validador("00")) == "00"
print("Todos los casos de prueba pasaron.")


## Ejercicio 5: Subconjunto con suma exacta

### Enunciado

Dada una lista de enteros positivos y un valor `objetivo`, determina si existe algún **subconjunto** de la lista cuya suma sea exactamente igual al objetivo. Resuélvelo probando todos los subconjuntos posibles, usando una técnica de "máscara de bits" (sin recursión).

### Modelo de pensamiento

1. A diferencia de los pares (ejercicio 1) o de los subarreglos **contiguos** (ejercicio 3), aquí un subconjunto puede tomar cualquier combinación de elementos, en cualquier posición, sin importar que sean contiguos. Eso cambia por completo cuántas posibilidades hay que revisar.
2. ¿Cuántos subconjuntos tiene una lista de `n` elementos? Cada elemento tiene exactamente dos estados posibles — "incluido" o "no incluido" — y esas decisiones son independientes entre sí → `2^n` subconjuntos posibles en total (incluyendo el vacío).
3. La clave para recorrerlos todos de forma sistemática, sin recursión, es representar cada subconjunto como un número binario de `n` bits: el bit en la posición `i` indica si el elemento `i` está incluido (1) o no (0). Recorriendo todos los números de 0 a `2^n - 1` con un solo `for mascara in range(2 ** n)`, se generan automáticamente todas las combinaciones de inclusión/exclusión, una vez cada una.
4. Para una máscara dada, hay que revisar bit por bit cuáles elementos están "prendidos": la operación `mascara & (1 << i)` indica si el bit `i` está encendido, y si lo está, se suma el elemento `i` correspondiente.
5. Esta es otra complejidad **exponencial**, pero ahora en función del **tamaño de la lista** (`n`), no de un parámetro externo como `k` en el ejercicio anterior — para listas de más de 25-30 elementos, esta fuerza bruta ya se vuelve impráctica.

### Complejidad

- **Tiempo:** O(2^n · n): hay 2^n máscaras posibles, y para cada una se revisan los `n` bits para calcular la suma correspondiente.
- **Espacio:** O(1) adicional (sin contar la lista de entrada): solo variables escalares para la máscara y la suma parcial de cada iteración.

In [ ]:
def existe_subconjunto_con_suma(numeros: list[int], objetivo: int) -> bool:
    ...

### Casos de prueba

In [ ]:
assert existe_subconjunto_con_suma([3, 34, 4, 12, 5, 2], 9) is True   # 4 + 5
assert existe_subconjunto_con_suma([3, 34, 4, 12, 5, 2], 30) is False
assert existe_subconjunto_con_suma([1, 2, 3], 0) is True   # el subconjunto vacío
assert existe_subconjunto_con_suma([5], 5) is True
print("Todos los casos de prueba pasaron.")


## Ejercicio 6: Contar soluciones del problema de las N reinas (reto)

### Enunciado

Dado un tablero de `n x n`, cuenta cuántas formas hay de colocar `n` reinas de modo que ninguna ataque a otra (ni misma fila, ni misma columna, ni misma diagonal). Resuélvelo por fuerza bruta **sin recursión**, aprovechando que cada solución candidata (una reina por fila y por columna) corresponde a una permutación de las columnas `0..n-1`: genera todas las permutaciones con `itertools.permutations` y valida solo la condición de las diagonales.

### Modelo de pensamiento

1. Sin ninguna restricción, colocar `n` reinas en un tablero de `n x n` en `n` casillas cualesquiera sería un problema con una cantidad de posibilidades gigantesca (elegir `n` casillas de `n²` posibles). Pero el enunciado ya regala una reducción importante: como no puede haber dos reinas en la misma fila **ni** en la misma columna, cada solución válida se describe con una sola lista de `n` números — en qué columna va la reina de cada fila — y esa lista es, ni más ni menos, una **permutación** de los números `0` a `n-1`.
2. Esto reduce el problema de "elegir n casillas de n²" a "probar todas las permutaciones de n columnas": hay `n!` (factorial de n) permutaciones posibles, en vez de una cantidad muchísimo mayor.
3. Para generarlas todas sin escribir una función recursiva propia, se usa `itertools.permutations(range(n))` — la librería estándar hace la enumeración internamente, sin que nuestro código recurra.
4. Cada permutación ya garantiza, por construcción, que no hay dos reinas en la misma fila ni en la misma columna. Solo falta comprobar la diagonal: dos reinas en las filas `i` y `j` (columnas `permutacion[i]` y `permutacion[j]`) se atacan diagonalmente si `abs(i - j) == abs(permutacion[i] - permutacion[j])`.
5. Para validar una permutación completa, compara todos los pares de filas `(i, j)` con `i < j` contra esa condición — si algún par falla, la permutación no es una solución válida.
6. Esta es la complejidad más costosa de los seis ejercicios: **factorial**, que crece incluso más rápido que exponencial. Por eso la fuerza bruta para N-reinas solo es viable para `n` pequeño; en la práctica, backtracking con poda (otra estrategia algorítmica) resuelve valores de `n` mucho mayores.

### Complejidad

- **Tiempo:** O(n! · n²): hay `n!` permutaciones, y validar cada una revisa hasta `n²/2` pares de filas para comprobar las diagonales.
- **Espacio:** O(n) adicional: `itertools.permutations` genera las permutaciones de a una (no las guarda todas en memoria a la vez), así que solo se necesita espacio para la permutación actual.

In [ ]:
from itertools import permutations


def contar_soluciones_n_reinas(n: int) -> int:
    ...


### Casos de prueba

In [ ]:
# Valores conocidos de la secuencia de N-reinas (OEIS A000170)
assert contar_soluciones_n_reinas(1) == 1
assert contar_soluciones_n_reinas(2) == 0
assert contar_soluciones_n_reinas(3) == 0
assert contar_soluciones_n_reinas(4) == 2
assert contar_soluciones_n_reinas(5) == 10
print("Todos los casos de prueba pasaron.")


## Para cerrar: cómo escaló la complejidad

| # | Ejercicio | Espacio de búsqueda | Tiempo |
|---|-----------|----------------------|--------|
| 1 | Par que suma un objetivo | pares de elementos | O(n²) |
| 2 | Búsqueda de patrón | posiciones de inicio | O(n·m) |
| 3 | Máxima suma de subarreglo | subarreglos contiguos | O(n²) |
| 4 | Descifrar código | códigos de k dígitos | O(k·10^k) |
| 5 | Subconjunto con suma exacta | subconjuntos | O(2^n·n) |
| 6 | N-reinas | permutaciones | O(n!·n²) |

La fuerza bruta siempre es **correcta** (revisa honestamente todas las posibilidades), pero su costo depende por completo de **qué** se está enumerando: recorrer pares o subarreglos de una lista de tamaño `n` da polinomios en `n`; recorrer códigos o subconjuntos da exponenciales; recorrer permutaciones da factoriales. Identificar a qué familia pertenece el espacio de búsqueda de un problema nuevo es, en sí mismo, una habilidad clave antes de intentar optimizarlo.